# Multi-Label Emotion Detection Using NLP

This notebook demonstrates the process of training models to detect multiple emotions in text.

## Overview
1. Data loading and exploration
2. Emotion distribution analysis
3. Text preprocessing
4. Traditional ML approach with Naive Bayes
5. Deep learning approach with BERT
6. Model evaluation and saving

In [1]:
# Import necessary libraries
import pandas as pd
import numpy as np
import nltk
import string
import torch
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.multiclass import OneVsRestClassifier
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from transformers import BertTokenizer, BertForSequenceClassification, Trainer, TrainingArguments

/Users/hsamadov/Library/Caches/pypoetry/virtualenvs/nlp-emotion-detection-JtJBIgvt-py3.13/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Data Loading and Initial Exploration

In [2]:
# Load the dataset
df = pd.read_csv(r"data/main.csv")

# Display the first few rows to understand the data structure
df.head()

,id,text,anger,fear,joy,sadness,surprise
0,eng_train_track_a_00001,"Colorado, middle of nowhere.",0,1,0,0,1
1,eng_train_track_a_00002,This involved swimming a pretty large lake tha...,0,1,0,0,0
2,eng_train_track_a_00003,It was one of my most shameful experiences.,0,1,0,1,0
3,eng_train_track_a_00004,"After all, I had vegetables coming out my ears...",0,0,0,0,0
4,eng_train_track_a_00005,Then the screaming started.,0,1,0,1,1


## 2. Emotion Distribution Analysis

Understanding the distribution of emotions in our dataset is crucial for identifying potential class imbalance issues that might affect model training.

In [3]:
# Calculate absolute counts for each emotion
emotion_counts = {
    'anger': df['anger'].sum(),
    'fear': df['fear'].sum(),
    'joy': df['joy'].sum(),
    'sadness': df['sadness'].sum(),
    'surprise': df['surprise'].sum()
}

total_entries = len(df)

# Calculate percentage distribution
emotion_percentages = {emotion: (count / total_entries) * 100 
                       for emotion, count in emotion_counts.items()}

# Display distribution statistics
print("Emotion Distribution Analysis")
print("=" * 30)
print(f"Total entries: {total_entries}\n")

print("Absolute counts:")
for emotion, count in emotion_counts.items():
    print(f"{emotion.capitalize()}: {count}")

print("\nPercentages:")
for emotion, percentage in emotion_percentages.items():
    print(f"{emotion.capitalize()}: {percentage:.2f}%")

Emotion Distribution Analysis
Total entries: 2768

Absolute counts:
Anger: 333
Fear: 1611
Joy: 674
Sadness: 878
Surprise: 839

Percentages:
Anger: 12.03%
Fear: 58.20%
Joy: 24.35%
Sadness: 31.72%
Surprise: 30.31%


## 3. Text Preprocessing

We'll prepare the text data for modeling by:
- Converting to lowercase
- Tokenizing
- Removing punctuation and stopwords
- Stemming words to their root form

In [4]:
# Download necessary NLTK resources
nltk.download('punkt')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /Users/hsamadov/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/hsamadov/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [5]:
# Initialize stopwords and stemmer
stop_words = set(stopwords.words('english'))
stemmer = PorterStemmer()

In [6]:
def preprocess(text):
    """
    Preprocess text for NLP tasks
    
    Args:
        text (str): Raw text input
        
    Returns:
        str: Preprocessed text with tokens joined by spaces
    """
    # Convert to lowercase
    text = text.lower()
    
    # Tokenize the text
    tokens = word_tokenize(text)
    
    # Remove punctuation and stopwords, then apply stemming
    tokens = [stemmer.stem(word) for word in tokens 
              if word not in string.punctuation and word not in stop_words]
    
    # Join tokens back into a single string
    return " ".join(tokens)

In [7]:
# Apply preprocessing to the text column
df["processed_text"] = df["text"].apply(preprocess)

## 4. Traditional ML Approach: Naive Bayes with OneVsRest

First, we'll implement a baseline model using a traditional machine learning approach:
- Convert text to bag-of-words features
- Train a multi-label classifier using Naive Bayes
- Evaluate the model performance

In [8]:
# Define our emotion labels
emotions = ['anger', 'fear', 'joy', 'sadness', 'surprise']

# Convert text to feature vectors
vectorizer = CountVectorizer()
X = vectorizer.fit_transform(df['processed_text'])
y = df[emotions]  # Multi-label target (n_samples, n_labels)

# Split data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# Initialize and train the model
# OneVsRest strategy for multi-label classification
model = OneVsRestClassifier(MultinomialNB())
model.fit(X_train, y_train)

# Generate predictions
y_pred = model.predict(X_test)

# Evaluate the model
print("Traditional ML (Naive Bayes) Results:")
print("-" * 40)
print(classification_report(y_test, y_pred, target_names=emotions))
print("-" * 40)
print("Exact Match Accuracy:", accuracy_score(y_test, y_pred))
print("-" * 40)

Traditional ML (Naive Bayes) Results:
----------------------------------------
              precision    recall  f1-score   support

       anger       0.26      0.10      0.14       105
        fear       0.68      0.79      0.73       496
         joy       0.45      0.26      0.33       185
     sadness       0.55      0.41      0.47       251
    surprise       0.56      0.34      0.42       264

   micro avg       0.60      0.50      0.54      1301
   macro avg       0.50      0.38      0.42      1301
weighted avg       0.56      0.50      0.52      1301
 samples avg       0.53      0.45      0.46      1301

----------------------------------------
Exact Match Accuracy: 0.23345367027677497
----------------------------------------


/Users/hsamadov/Library/Caches/pypoetry/virtualenvs/nlp-emotion-detection-JtJBIgvt-py3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/hsamadov/Library/Caches/pypoetry/virtualenvs/nlp-emotion-detection-JtJBIgvt-py3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/hsamadov/Library/Caches/pypoetry/virtualenvs/nlp-emotion-detection-JtJBIgvt-py3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined a

## 5. Deep Learning Approach: BERT for Multi-Label Classification

### 5.1 Prepare Data for BERT

In [9]:
# Define emotion labels
emotions = ['anger', 'fear', 'joy', 'sadness', 'surprise']

# Prepare features and labels
X = df['text'].tolist()  # Using original text instead of processed text for BERT
y = df[emotions].values

In [10]:
# Initialize BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

In [11]:
# Create a custom dataset for BERT
class EmotionDataset(torch.utils.data.Dataset):
    """
    PyTorch dataset for emotion detection with BERT
    """
    def __init__(self, texts, labels, tokenizer, max_len=128):
        """
        Initialize the dataset
        
        Args:
            texts (list): List of text samples
            labels (np.array): Multi-hot encoded emotion labels
            tokenizer: BERT tokenizer
            max_len (int): Maximum sequence length
        """
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        # Tokenize and prepare inputs for BERT
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        
        # Prepare item dictionary with all required inputs
        item = {key: val.squeeze() for key, val in encoding.items()}
        item['labels'] = torch.tensor(label, dtype=torch.float)
        return item

In [12]:
# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Create dataset objects
train_dataset = EmotionDataset(X_train, y_train, tokenizer)
test_dataset = EmotionDataset(X_test, y_test, tokenizer)

### 5.2 Initialize BERT Model

In [13]:
# Initialize BERT model for multi-label classification
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=len(emotions),
    problem_type="multi_label_classification"
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


### 5.3 Configure Training Arguments

In [14]:
# Define training arguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=2,
    per_device_train_batch_size=8,
    eval_strategy="epoch",  # Changed from evaluation_strategy
    save_strategy="epoch",
    logging_dir='./logs',
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
)

### 5.4 Train the Model

In [15]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset
)


In [16]:
# Train the model
trainer.train()

/Users/hsamadov/Library/Caches/pypoetry/virtualenvs/nlp-emotion-detection-JtJBIgvt-py3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Epoch,Training Loss,Validation Loss
1,No log,0.414027
2,0.394600,0.375422


/Users/hsamadov/Library/Caches/pypoetry/virtualenvs/nlp-emotion-detection-JtJBIgvt-py3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=554, training_loss=0.3824876602806339, metrics={'train_runtime': 146.2072, 'train_samples_per_second': 30.286, 'train_steps_per_second': 3.789, 'total_flos': 291271783707648.0, 'train_loss': 0.3824876602806339, 'epoch': 2.0})

### 5.5 Evaluate the BERT Model

In [17]:
# Generate predictions
preds = trainer.predict(test_dataset)

# Convert logits to probabilities and then to binary predictions
y_pred_probs = torch.sigmoid(torch.tensor(preds.predictions)).numpy()
y_pred = (y_pred_probs >= 0.5).astype(int)

# Evaluate the model
print("BERT Model Results:")
print("-" * 40)
print(classification_report(y_test, y_pred, target_names=emotions))
print("-" * 40)
print("Exact Match Accuracy:", accuracy_score(y_test, y_pred))
print("-" * 40)

/Users/hsamadov/Library/Caches/pypoetry/virtualenvs/nlp-emotion-detection-JtJBIgvt-py3.13/lib/python3.13/site-packages/torch/utils/data/dataloader.py:683: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, then device pinned memory won't be used.
  warnings.warn(warn_msg)


BERT Model Results:
----------------------------------------
              precision    recall  f1-score   support

       anger       0.74      0.32      0.45        72
        fear       0.78      0.87      0.82       330
         joy       0.71      0.57      0.63       115
     sadness       0.71      0.65      0.68       167
    surprise       0.82      0.63      0.71       179

   micro avg       0.77      0.69      0.73       863
   macro avg       0.75      0.61      0.66       863
weighted avg       0.76      0.69      0.72       863
 samples avg       0.67      0.63      0.63       863

----------------------------------------
Exact Match Accuracy: 0.4223826714801444
----------------------------------------


/Users/hsamadov/Library/Caches/pypoetry/virtualenvs/nlp-emotion-detection-JtJBIgvt-py3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/hsamadov/Library/Caches/pypoetry/virtualenvs/nlp-emotion-detection-JtJBIgvt-py3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in samples with no true labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/hsamadov/Library/Caches/pypoetry/virtualenvs/nlp-emotion-detection-JtJBIgvt-py3.13/lib/python3.13/site-packages/sklearn/metrics/_classification.py:1706: UndefinedMetricWarning: F-score is ill-defined a

## 6. Save the BERT Model

We'll save the trained model for later use in production.

In [18]:
# Save the model properly in a format that can be loaded by from_pretrained
model_save_path = './results'
trainer.save_model(model_save_path)
print(f"Model saved to {model_save_path}")

# Verify the saved model
print("Files saved in the model directory:")
for file in os.listdir(model_save_path):
    print(f"- {file}")

# Test loading the model
test_model = BertForSequenceClassification.from_pretrained(model_save_path)
print("Model loaded successfully!")
print("The model can now be used in production.")

Model saved to ./results
Files saved in the model directory:
- model.safetensors
- checkpoint-277
- checkpoint-554
- checkpoint-500
- config.json
- training_args.bin
Model loaded successfully!
The model can now be used in production.


## Conclusion

In this notebook, we've:
1. Analyzed the distribution of emotions in our dataset
2. Preprocessed text data for NLP tasks
3. Built a baseline model using Naive Bayes
4. Implemented a more advanced BERT-based model
5. Compared the performance of both approaches
6. Saved the BERT model for production use

The BERT model typically offers superior performance due to its ability to understand context and semantic meaning, though at the cost of higher computational requirements.